# Week 11: Error Handling — PHASE 3: Build a checkable report

*📚 Computer Programming I · ⏱️ 5 Hours · 👨‍🏫 Dr. Arif Solmaz*

## Separate a missing reading from zero

Our functions work when their inputs are what they expect. Real incoming text also contains blanks and failure messages. We need a clear outcome for those cases.

A teaching temperature stream contains “0”, “-12.5”, a blank and “ERROR”. Valid measurements must be finite numbers between −50 and 60 °C, including both limits.

**Try this first — before code.** Sort the four records into valid readings and rejected records. If a conversion fails, would replacing it with zero preserve what happened?

**Why this week's tool?** try/except handles a failed conversion. Separate checks handle nonfinite or out-of-range numbers. Each rejection comes with a reason.

**By the end.** Keep valid zero and negative temperatures. Keep them separate from records that could not be measured or read.


## Explain the program: Invalid data is not the same as zero

Error handling must keep the meaning of the valid readings. A failed conversion, a nonfinite number and an out-of-range reading each need their own handling. Replacing them with zero invents measurements that never happened.

**Draw or trace.** Trace four inputs: "0", "20", "missing" and "nan". Check conversion, then finiteness, then the allowed range. Only then add a value to the summary.

**Predict before running.** Suppose missing data becomes zero. How does the average of the valid readings 0 and 20 change?

<details><summary>Trace and explanation — after your prediction</summary>

1. With only the two valid readings, total = 20 and count = 2, giving mean 10.
2. Replacing one missing value with zero makes count 3 and gives about 6.67.
3. A float conversion of "nan" succeeds. The result still needs a finiteness check.

Zero is valid when the sensor rules allow it. Missing data is not a zero reading. Catch the relevant exception, and record or report the failure. Decide what to do when every row is invalid, without dividing by zero.

</details>

**Change one thing.** Change every row to invalid data. Write the report’s no-data message before running the program.

**Türkçe:** Eksik veriyi sıfır yapmak ölçüm uydurur ve ortalamayı değiştirir. Dönüşüm, sonluluk ve aralık denetimleri ayrı adımlardır.


<details><summary>Learning objectives</summary>

## 🎯 Learning Objectives

By the end of this week, you will be able to:

- Distinguish between **syntax errors**, **runtime errors**, and **logical errors**
- Recognize common Python error types (`TypeError`, `ValueError`, `IndexError`, etc.)
- Use **`try/except`** blocks to catch and handle errors gracefully
- Catch **specific error types** with multiple `except` blocks
- Use **`else`** and **`finally`** clauses for complete error handling
- Implement **input validation** with retry patterns
- Use **`raise`** to create your own errors
- Apply **defensive programming** techniques to build robust functions

</details>


<details><summary>Class participation and assessment</summary>

---
## 🤝 Engineering Learning Contract

- **Professional relevance:** examples and core exercises model the data, sensing, automation, numerical, and decision tasks used in engineering.
- **Interaction:** predict before running, compare reasoning with a partner, and ask whenever a step is unclear; scheduled checkpoints guarantee question time.
- **Assessment alignment:** worked examples and Core Exercises 1–8 rehearse the same reasoning operations used on exams—trace, implement, debug, interpret, and justify—while exam values and contexts may change.
- **Assessment:** Assessment consists only of the midterm exam (50%) and final exam (50%). Weekly notebooks, exercises, projects, demonstrations and presentations are ungraded practice; no weekly submission is required.

</details>


<details><summary>Class schedule and checkpoints</summary>

---
## 🧭 Five-Hour Class Roadmap

This notebook is designed for one five-hour class with four short breaks.

| Target | Activity |
|---|---|
| 00:00–00:55 | Concepts and examples → Checkpoint 1 |
| 00:55–01:05 | Break |
| 01:05–01:55 | Concepts and examples → Checkpoint 2 |
| 01:55–02:05 | Break |
| 02:05–02:55 | Concepts and examples → Checkpoint 3 |
| 02:55–03:05 | Break |
| 03:05–03:55 | Concepts and examples → Checkpoint 4 |
| 03:55–04:05 | Break |
| 04:05–04:45 | Core Practice (Exercises 1–8) → Checkpoint 5 |
| 04:45–05:00 | Review and retry failed checks |

Concept checkpoints compare your predictions with an expected answer. Checkpoint 5 is your practice reflection. No grading submission is sent by these tools. Save the notebook to keep your work. Exercises 9 and above are optional extensions.

</details>


In [ ]:
#@title Study tools — run this cell once (the code is hidden; you do not need to read it)
# These small tools give local study feedback.
_checkpoint_results = {}

def check_answer(number, answer, expected, explanation):
    actual = str(answer).strip().lower().replace(" ", "")
    target = str(expected).strip().lower().replace(" ", "")
    correct = actual == target
    _checkpoint_results[int(number)] = ("Concept check", int(correct), 1)
    if correct:
        print(f"Checkpoint {number}: correct. {explanation}")
    elif not str(answer).strip():
        print(f"Checkpoint {number}: enter your prediction, then run again.")
    else:
        print(f"Checkpoint {number}: review the example and try again.")
    return correct

def record_checkpoint(number, checks):
    """Report each concrete concept check used by the introductory notebook."""
    passed = sum(bool(correct) for _, correct in checks)
    _checkpoint_results[int(number)] = ("Concept checks", passed, len(checks))
    print(f"Checkpoint {number}: {passed}/{len(checks)} concept checks match.")
    for label, correct in checks:
        print(("OK: " if correct else "Review: ") + label)
    return passed, len(checks)

def exercise_checkpoint(number, practiced, expected=8):
    """Summarize an explicit self-report; this does not grade your code."""
    if not isinstance(practiced, (list, tuple, set)):
        _checkpoint_results.pop(int(number), None)
        print("Use a list of exercise numbers, for example [1, 2].")
        return 0, expected
    if any(type(item) is not int or not 1 <= item <= expected for item in practiced):
        _checkpoint_results.pop(int(number), None)
        print(f"Use whole exercise numbers from 1 to {expected}.")
        return 0, expected
    done = set(practiced)
    _checkpoint_results[int(number)] = ("Practice self-report", len(done), expected)
    print(f"Practice self-report: {len(done)}/{expected} core exercises reviewed.")
    print("This is your reflection, not a correctness score or a grade.")
    remaining = [str(i) for i in range(1, expected + 1) if i not in done]
    if remaining:
        print("Still to review:", ", ".join(remaining))
    print("For each exercise: test the result, explain the steps, then compare with the worked solution.")
    return len(done), expected

def show_progress_summary():
    print("\nMy study feedback (this runtime)")
    for number in range(1, 6):
        if number in _checkpoint_results:
            kind, count, total = _checkpoint_results[number]
            print(f"{number}. {kind}: {count}/{total}")
        else:
            print(f"{number}. Not run yet")
    print("These checks send no grading submission. Save your notebook to keep your work.")

print("Local study tools ready.")


## Joining this lesson: records and returned pairs

Recall a small record: `student = {"name": "Elif", "scores": [85, 92]}`.
`student["name"]` reads a key; `student["average"] = 88.5` adds a value;
`"scores" in student` checks whether the key exists. To display the entries, use
`for key, value in student.items(): print(key, value)`.

A function can `return True, "OK"`. Its caller writes `valid, reason = validate(...)`
to unpack the returned tuple. Keep the order consistent: first the decision, then
the explanation. Dictionary keys name fields; tuple positions preserve a short,
agreed order. Dictionaries were introduced in Week 7 and returned pairs in Weeks 9–10.

**Türkçe:** Sözlük anahtarı alanın adıdır; demet açma iki sonucu sırayla alır.
Bir kaydı okumadan önce gerekli alanların bulunduğunu kontrol edin.


---
## Part 1: What Are Errors?

Errors are a natural part of programming. Python has three main types:

| Error Type | When It Happens | Example |
|------------|----------------|----------|
| **Syntax Error** | Before running — code is malformed | `print("hello"` (missing `)`) |
| **Runtime Error** | During running — something goes wrong | `10 / 0` (division by zero) |
| **Logical Error** | Code runs fine but gives wrong result | Using `+` instead of `*` |

**Figure 1.1: Syntax error — Python cannot even start running**

In [2]:
# Uncomment the line below to see a SyntaxError:
# print("Hello World"

# Python tells you EXACTLY where the problem is:
# SyntaxError: unexpected EOF while parsing
print("This cell runs fine because the error is commented out.")

This cell runs fine because the error is commented out.


**Figure 1.2: Runtime error — code is valid but fails during execution**

In [3]:
# This code is syntactically correct, but will crash:
numerator = 10
denominator = 0

try:
    result = numerator / denominator
except ZeroDivisionError:
    print("Runtime Error: Cannot divide by zero!")
    print("The program crashed because of a ZeroDivisionError.")

Runtime Error: Cannot divide by zero!
The program crashed because of a ZeroDivisionError.


**Figure 1.3: Logical error — no crash, but wrong answer**

In [4]:
# Calculate the area of a rectangle
width = 5
height = 10

# BUG: Using + instead of *
area_wrong = width + height   # Logical error! Should be *
area_correct = width * height

print(f"Wrong area: {area_wrong}")    # 15 (no crash, but wrong!)
print(f"Correct area: {area_correct}")  # 50

Wrong area: 15
Correct area: 50


> 💡 **Note:** Syntax errors are caught **before** the program runs. Runtime errors crash **during** execution. Logical errors are the hardest to find because the program runs without crashing — it just gives the wrong answer!

---
## Part 2: Common Python Errors

Here are the most common runtime errors you'll encounter:

| Error | When It Happens | Example |
|-------|----------------|----------|
| `TypeError` | Wrong type for an operation | `"5" + 3` |
| `ValueError` | Right type, wrong value | `int("hello")` |
| `IndexError` | List index out of range | `[1,2,3][5]` |
| `ZeroDivisionError` | Dividing by zero | `10 / 0` |
| `NameError` | Variable doesn't exist | `print(xyz)` |
| `KeyError` | Dictionary key doesn't exist | `{"a": 1}["b"]` |

**Figure 2.1: Demonstrating each common error type**

In [5]:
# Let's see each error in action (safely, using try/except)

def demo_type_error():
    return "5" + 3

def demo_value_error():
    return int("hello")

def demo_index_error():
    return [1, 2, 3][10]

def demo_zero_division():
    return 10 / 0

def demo_key_error():
    return {"name": "Ali"}["age"]

# Each pair holds the error name and the function that causes it.
errors_to_demo = [
    ("TypeError",          demo_type_error),
    ("ValueError",         demo_value_error),
    ("IndexError",         demo_index_error),
    ("ZeroDivisionError",  demo_zero_division),
    ("KeyError",           demo_key_error),
]

for name, action in errors_to_demo:
    try:
        action()
    except Exception as e:
        print(f"{name:22s} → {e}")

TypeError              → can only concatenate str (not "int") to str
ValueError             → invalid literal for int() with base 10: 'hello'
IndexError             → list index out of range
ZeroDivisionError      → division by zero
KeyError               → 'age'


**Figure 2.2: Reading error messages — they tell you what went wrong**

In [6]:
# Error messages have useful information:
grades = [85, 92, 78, 65]

try:
    print(grades[10])   # Only indices 0-3 exist!
except IndexError as e:
    print(f"Error type: IndexError")
    print(f"Error message: {e}")
    print(f"List length: {len(grades)}")
    print(f"Valid indices: 0 to {len(grades) - 1}")

Error type: IndexError
Error message: list index out of range
List length: 4
Valid indices: 0 to 3


> 💡 **Note:** Always **read the error message carefully**. Python tells you the error type, the message, and the exact line number where it happened. This is your best debugging tool!

---
### ⏱️ Checkpoint 1 of 5 — Error types (target 00:55)

Is dividing by zero a syntax error or runtime error?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [7]:
checkpoint_1_answer = ""  # enter your answer
check_answer(
    1, checkpoint_1_answer, 'runtime',
    'The code parses but fails while executing.',
)


Checkpoint 1: enter your prediction, then run again.


False

---
## Part 3: `try/except` Basics

The `try/except` block lets you **catch errors** and handle them gracefully instead of crashing.

```python
try:
    # Code that might cause an error
    risky_operation()
except:
    # Code that runs if an error occurs
    handle_error()
```

**Figure 3.1: Basic try/except — preventing a crash**

In [8]:
# WITHOUT try/except — program crashes
# result = 10 / 0   # ZeroDivisionError — everything after this stops!

# WITH try/except — program continues
try:
    result = 10 / 0
except:
    print("Something went wrong!")
    result = 0

print(f"Result: {result}")   # Result: 0
print("Program continues normally!")   # This line runs!

Something went wrong!
Result: 0
Program continues normally!


**Figure 3.2: Safe division function**

In [9]:
def safe_divide(a, b):
    """Divide a by b safely."""
    try:
        return a / b
    except:
        print(f"Cannot divide {a} by {b}")
        return None

print(safe_divide(10, 3))    # 3.3333...
print(safe_divide(10, 0))    # Cannot divide 10 by 0 → None
print(safe_divide(10, "x"))  # Cannot divide 10 by x → None

3.3333333333333335
Cannot divide 10 by 0
None
Cannot divide 10 by x
None


**Figure 3.3: Safe user input**

In [10]:
def get_number(text):
    """Simulate getting a number from user input."""
    try:
        return float(text)
    except:
        print(f"'{text}' is not a valid number.")
        return None

# Simulating different user inputs:
print(get_number("42"))       # 42.0
print(get_number("3.14"))     # 3.14
print(get_number("hello"))    # 'hello' is not a valid number. → None
print(get_number(""))         # '' is not a valid number. → None

42.0
3.14
'hello' is not a valid number.
None
'' is not a valid number.
None


> 💡 **Note:** Using a bare `except:` catches ALL errors. This is convenient but can hide bugs. It's better to catch **specific** error types, as we'll see next.

---
## Part 4: Catching Specific Errors

You can catch **specific error types** to handle each one differently. This is much better than catching everything!

**Figure 4.1: Multiple except blocks**

In [11]:
def safe_divide_v2(a, b):
    """Divide a by b with specific error handling."""
    try:
        result = a / b
        return result
    except ZeroDivisionError:
        print("Error: Cannot divide by zero!")
        return None
    except TypeError:
        print(f"Error: Cannot divide {type(a).__name__} by {type(b).__name__}")
        return None

print(safe_divide_v2(10, 3))      # 3.333...
print(safe_divide_v2(10, 0))      # Error: Cannot divide by zero!
print(safe_divide_v2("10", 2))    # Error: Cannot divide str by int

3.3333333333333335
Error: Cannot divide by zero!
None
Error: Cannot divide str by int
None


**Figure 4.2: Accessing the error message with `as`**

In [12]:
def process_grade(value):
    """Convert a value to a grade (0-100)."""
    try:
        grade = int(value)
        if grade < 0 or grade > 100:
            print(f"Grade {grade} is out of range (0-100)")
            return None
        return grade
    except ValueError as e:
        print(f"Cannot convert to number: {e}")
        return None
    except TypeError as e:
        print(f"Wrong type: {e}")
        return None

print(process_grade("85"))     # 85
print(process_grade("abc"))    # Cannot convert to number: ...
print(process_grade(None))     # Wrong type: ...
print(process_grade("150"))    # Grade 150 is out of range

85
Cannot convert to number: invalid literal for int() with base 10: 'abc'
None
Wrong type: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
None
Grade 150 is out of range (0-100)
None


**Figure 4.3: Catching multiple error types in one line**

In [13]:
def get_item(data, index):
    """Safely get an item from a list or dictionary."""
    try:
        return data[index]
    except (IndexError, KeyError, TypeError) as e:
        print(f"Cannot access item: {e}")
        return None

# Works with lists:
print(get_item([10, 20, 30], 1))     # 20
print(get_item([10, 20, 30], 10))    # Cannot access item: ...

# Works with dictionaries:
print(get_item({"name": "Elif"}, "name"))  # Elif
print(get_item({"name": "Elif"}, "age"))   # Cannot access item: ...

20
Cannot access item: list index out of range
None
Elif
Cannot access item: 'age'
None


> 💡 **Note:** Always catch the **most specific** error first. Python checks `except` blocks from top to bottom and uses the first one that matches.

---
### ⏱️ Checkpoint 2 of 5 — Exceptions (target 01:55)

Which exception does `int('abc')` raise?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [14]:
checkpoint_2_answer = ""  # enter your answer
check_answer(
    2, checkpoint_2_answer, 'ValueError',
    'The string has the wrong value format for integer conversion.',
)


Checkpoint 2: enter your prediction, then run again.


False

---
## Part 5: The `else` Clause

The `else` block runs **only if no error occurred** in the `try` block. It's useful for code that should only execute on success.

```python
try:
    # Risky code
except SomeError:
    # Handle error
else:
    # Runs only if NO error occurred
```

**Figure 5.1: Using else for success-only code**

In [15]:
def divide_and_report(a, b):
    """Divide and print the result only on success."""
    try:
        result = a / b
    except ZeroDivisionError:
        print("Division by zero is not allowed.")
    else:
        # This only runs if division succeeded
        print(f"{a} / {b} = {result:.2f}")
        if result > 10:
            print("That's a big number!")

divide_and_report(100, 3)    # 100 / 3 = 33.33 → That's a big number!
divide_and_report(10, 0)     # Division by zero is not allowed.
divide_and_report(6, 2)      # 6 / 2 = 3.00

100 / 3 = 33.33
That's a big number!
Division by zero is not allowed.
6 / 2 = 3.00


**Figure 5.2: Why use else instead of putting code in try?**

In [16]:
def convert_and_double(text):
    """Convert text to number and double it."""
    try:
        number = float(text)
    except ValueError:
        print(f"'{text}' is not a number.")
    else:
        # Only runs if conversion succeeded
        # If this code had a bug, it wouldn't be caught by
        # the except ValueError above — which is what we want!
        doubled = number * 2
        print(f"{text} doubled is {doubled}")

convert_and_double("7.5")    # 7.5 doubled is 15.0
convert_and_double("abc")    # 'abc' is not a number.

7.5 doubled is 15.0
'abc' is not a number.


> 💡 **Note:** The `else` clause keeps your `try` block small and focused. Only put the **risky** code in `try` — put the **success** code in `else`. This prevents accidentally catching errors from the wrong place.

---
## Part 6: The `finally` Clause

The `finally` block **always runs**, whether an error occurred or not. It's used for cleanup code.

```python
try:
    # Risky code
except SomeError:
    # Handle error
else:
    # On success
finally:
    # ALWAYS runs — cleanup code
```

**Figure 6.1: finally always runs**

In [17]:
def process_data(value):
    """Process data with full error handling."""
    print(f"\nProcessing: {value}")
    try:
        result = 100 / value
    except ZeroDivisionError:
        print("  Error: division by zero")
    except TypeError:
        print(f"  Error: cannot divide by {type(value).__name__}")
    else:
        print(f"  Result: {result:.2f}")
    finally:
        print("  Processing complete.")   # Always prints!

process_data(4)       # Success path
process_data(0)       # ZeroDivisionError path
process_data("abc")   # TypeError path


Processing: 4
  Result: 25.00
  Processing complete.

Processing: 0
  Error: division by zero
  Processing complete.

Processing: abc
  Error: cannot divide by str
  Processing complete.


**Figure 6.2: Full error handling structure summary**

In [18]:
# The complete try/except/else/finally structure:

print("=== Full Structure Demo ===")
numbers = [10, 0, "five", 4]

for num in numbers:
    print(f"\nTrying 20 / {num}:")
    try:
        result = 20 / num
    except ZeroDivisionError:
        print("  EXCEPT: Can't divide by zero")
    except TypeError:
        print(f"  EXCEPT: '{num}' is not a number")
    else:
        print(f"  ELSE: Result = {result:.1f}")
    finally:
        print("  FINALLY: Moving on...")

=== Full Structure Demo ===

Trying 20 / 10:
  ELSE: Result = 2.0
  FINALLY: Moving on...

Trying 20 / 0:
  EXCEPT: Can't divide by zero
  FINALLY: Moving on...

Trying 20 / five:
  EXCEPT: 'five' is not a number
  FINALLY: Moving on...

Trying 20 / 4:
  ELSE: Result = 5.0
  FINALLY: Moving on...


> 💡 **Note:** `finally` is most useful for cleanup tasks like closing files or database connections. Even if an error crashes your program, `finally` still runs.

---
### ⏱️ Checkpoint 3 of 5 — else (target 02:55)

When does a `try` statement's `else` block run? Answer: success or failure.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [19]:
checkpoint_3_answer = ""  # enter your answer
check_answer(
    3, checkpoint_3_answer, 'success',
    '`else` runs when no exception occurs.',
)


Checkpoint 3: enter your prediction, then run again.


False

---
## Part 7: Input Validation with `try/except`

One of the most common uses of `try/except` is **validating user input**. Users often type unexpected things!

**Figure 7.1: Safe integer input function**

In [20]:
def safe_int_input(prompt):
    """Keep asking until the user enters a valid integer."""
    while True:
        text = input(prompt)
        try:
            return int(text)
        except ValueError:
            print(f"'{text}' is not a valid integer. Try again.")

# Uncomment to test interactively:
# age = safe_int_input("Enter your age: ")
# print(f"Your age is {age}")

# Let's demonstrate with simulated inputs:
test_inputs = ["abc", "3.14", "25"]
for t in test_inputs:
    try:
        result = int(t)
        print(f"'{t}' → {result} ✓")
    except ValueError:
        print(f"'{t}' → Not a valid integer ✗")

'abc' → Not a valid integer ✗
'3.14' → Not a valid integer ✗
'25' → 25 ✓


**Figure 7.2: Validated input with range checking**

In [21]:
def get_grade_input(prompt, min_val=0, max_val=100):
    """Get a valid grade from user (simulated here)."""
    # Simulating with test values
    test_values = ["abc", "-5", "150", "85"]
    
    for text in test_values:
        print(f"  Input: '{text}'", end=" → ")
        try:
            value = int(text)
        except ValueError:
            print(f"Not a number, try again.")
            continue
        
        if value < min_val or value > max_val:
            print(f"Must be between {min_val} and {max_val}, try again.")
            continue
        
        print(f"Valid! Grade = {value}")
        return value

print("Enter a grade (0-100):")
grade = get_grade_input("Grade: ")

Enter a grade (0-100):
  Input: 'abc' → Not a number, try again.
  Input: '-5' → Must be between 0 and 100, try again.
  Input: '150' → Must be between 0 and 100, try again.
  Input: '85' → Valid! Grade = 85


**Figure 7.3: The retry pattern — ask up to N times**

In [22]:
def get_number_with_retries(prompt, max_attempts=3):
    """Ask for a number with limited retries."""
    # Simulated inputs for demonstration
    simulated = ["hello", "world", "42"]
    
    for attempt in range(1, max_attempts + 1):
        text = simulated[attempt - 1] if attempt <= len(simulated) else "0"
        print(f"  Attempt {attempt}/{max_attempts}: '{text}'", end=" → ")
        try:
            number = float(text)
            print(f"Valid! Got {number}")
            return number
        except ValueError:
            remaining = max_attempts - attempt
            if remaining > 0:
                print(f"Invalid. {remaining} attempt(s) remaining.")
            else:
                print("Invalid. No attempts remaining.")
    
    print("  Too many failed attempts!")
    return None

result = get_number_with_retries("Enter a number: ")
print(f"Final result: {result}")

  Attempt 1/3: 'hello' → Invalid. 2 attempt(s) remaining.
  Attempt 2/3: 'world' → Invalid. 1 attempt(s) remaining.
  Attempt 3/3: '42' → Valid! Got 42.0
Final result: 42.0


> 💡 **Note:** The retry pattern is very useful in real programs. Always set a **maximum number of attempts** to avoid infinite loops if something goes wrong.

---
## Part 8: Raising Errors

You can **create your own errors** using `raise`. This is useful when your function receives invalid arguments.

**Figure 8.1: Using raise to signal problems**

In [23]:
def set_age(age):
    """Set a person's age with validation."""
    if not isinstance(age, int):
        raise TypeError("Age must be an integer")
    if age < 0 or age > 150:
        raise ValueError(f"Age must be between 0 and 150, got {age}")
    return age

# Test valid input:
print(set_age(25))   # 25

# Test invalid inputs:
try:
    set_age(-5)
except ValueError as e:
    print(f"Caught: {e}")   # Caught: Age must be between 0 and 150, got -5

try:
    set_age("twenty")
except TypeError as e:
    print(f"Caught: {e}")   # Caught: Age must be an integer

25
Caught: Age must be between 0 and 150, got -5
Caught: Age must be an integer


**Figure 8.2: Using raise in a library function**

In [24]:
def calculate_bmi(weight_kg, height_m):
    """Calculate BMI with input validation."""
    if weight_kg <= 0:
        raise ValueError(f"Weight must be positive, got {weight_kg}")
    if height_m <= 0:
        raise ValueError(f"Height must be positive, got {height_m}")
    if height_m > 3:
        raise ValueError(f"Height {height_m}m seems too large. Did you mean centimeters?")
    return weight_kg / (height_m ** 2)

# Valid:
print(f"BMI: {calculate_bmi(75, 1.78):.1f}")   # BMI: 23.7

# Invalid — helpful error messages:
try:
    calculate_bmi(75, 178)   # Oops, used cm instead of m!
except ValueError as e:
    print(f"Error: {e}")   # Height 178m seems too large...

BMI: 23.7
Error: Height 178m seems too large. Did you mean centimeters?


> 💡 **Note:** Use `raise` to catch mistakes **early** — before they cause confusing errors later. A clear error message like "Height must be positive" is much better than a cryptic "ZeroDivisionError" somewhere deep in the code.

---
## Part 9: Defensive Programming

There are two main philosophies for handling potential errors:

| Approach | Name | Style |
|----------|------|-------|
| **LBYL** | Look Before You Leap | Check conditions first, then act |
| **EAFP** | Easier to Ask Forgiveness than Permission | Try it, handle errors if they occur |

Python generally prefers **EAFP** (using `try/except`), but both are useful.

**Figure 9.1: LBYL vs EAFP comparison**

In [25]:
student = {"name": "Canan", "grades": [85, 92, 78]}

# LBYL — Look Before You Leap
def get_average_lbyl(student):
    if "grades" in student:           # Check first
        if len(student["grades"]) > 0:  # Check again
            return sum(student["grades"]) / len(student["grades"])
    return None

# EAFP — Easier to Ask Forgiveness than Permission
def get_average_eafp(student):
    try:
        return sum(student["grades"]) / len(student["grades"])
    except (KeyError, ZeroDivisionError, TypeError):
        return None

# Both work the same:
print(f"LBYL: {get_average_lbyl(student):.1f}")  # 85.0
print(f"EAFP: {get_average_eafp(student):.1f}")  # 85.0

# Both handle missing data:
print(f"LBYL: {get_average_lbyl({})}")  # None
print(f"EAFP: {get_average_eafp({})}")  # None

LBYL: 85.0
EAFP: 85.0
LBYL: None
EAFP: None


**Figure 9.2: When LBYL is clearer**

In [26]:
# Sometimes checking first is cleaner:

def safe_average(numbers):
    """Calculate average with validation."""
    # LBYL — clear and readable
    if not numbers:               # Empty list?
        return 0
    if not isinstance(numbers, list):  # Not a list?
        return 0
    return sum(numbers) / len(numbers)

print(safe_average([10, 20, 30]))  # 20.0
print(safe_average([]))            # 0
print(safe_average(None))          # 0

20.0
0
0


> 💡 **Note:** Use **LBYL** when the check is simple and readable. Use **EAFP** when there are many things that could go wrong. In Python, `try/except` is generally preferred because it's often cleaner.

---
## Part 10: Building Robust Functions

Let's combine everything we've learned to build **robust** functions that handle errors gracefully.

**Figure 10.1: A robust calculator function**

In [27]:
def calculator(a, b, operation):
    """A robust calculator that handles all errors."""
    # Validate inputs
    try:
        a = float(a)
        b = float(b)
    except (ValueError, TypeError):
        return "Error: Both values must be numbers"
    
    # Perform operation
    if operation == "+":
        return a + b
    elif operation == "-":
        return a - b
    elif operation == "*":
        return a * b
    elif operation == "/":
        if b == 0:
            return "Error: Cannot divide by zero"
        return a / b
    elif operation == "**":
        return a ** b
    else:
        return f"Error: Unknown operation '{operation}'"

# Test all cases:
print(calculator(10, 3, "+"))      # 13.0
print(calculator(10, 3, "/"))      # 3.333...
print(calculator(10, 0, "/"))      # Error: Cannot divide by zero
print(calculator("abc", 3, "+"))   # Error: Both values must be numbers
print(calculator(2, 8, "**"))      # 256.0
print(calculator(10, 3, "&"))      # Error: Unknown operation '&'

13.0
3.3333333333333335
Error: Cannot divide by zero
Error: Both values must be numbers
256.0
Error: Unknown operation '&'


### Conversion is only the first validation step

`float("nan")` and `float("inf")` succeed. They do not describe usable finite grades.
For NaN, both `score < 0` and `score > 100` are false, so that pair of comparisons
cannot reject it. After conversion, use `math.isfinite(score)`, then check the range.
`import math` loads Python's standard maths tools; no installation is required.

**Türkçe:** Sayıya çevrilebilmek, geçerli not olmak demek değildir. Önce sonlu
olduğunu, sonra 0–100 aralığında olduğunu kontrol edin.


**Figure 10.2: A robust data processing pipeline**

In [28]:
import math

def process_student_data(raw_data):
    """Process a list of student score strings into a report."""
    valid_scores = []
    errors = []
    
    for i, item in enumerate(raw_data):
        try:
            score = float(item)
        except (ValueError, TypeError):
            errors.append(f"Item {i}: '{item}' is not a number")
            continue
        
        if not math.isfinite(score):
            errors.append(f"Item {i}: {score} is not finite")
            continue

        if score < 0 or score > 100:
            errors.append(f"Item {i}: {score} is out of range (0-100)")
            continue
        
        valid_scores.append(score)
    
    # Report
    print(f"Processed {len(raw_data)} items:")
    print(f"  Valid: {len(valid_scores)}")
    print(f"  Errors: {len(errors)}")
    
    if errors:
        print("\nError details:")
        for err in errors:
            print(f"  - {err}")
    
    if valid_scores:
        avg = sum(valid_scores) / len(valid_scores)
        print(f"\nAverage of valid scores: {avg:.1f}")
    
    return valid_scores

# Test with messy data:
data = ["85", "92", "abc", "78", "-5", "110", "65", None, "90"]
clean = process_student_data(data)
print("\nNonfinite example:")
process_student_data(["85", "nan", "92", "inf"])
# Expected: 2 valid, 2 errors, average 88.5.


Processed 9 items:


  Valid: 5
  Errors: 4

Error details:
  - Item 2: 'abc' is not a number
  - Item 4: -5.0 is out of range (0-100)
  - Item 5: 110.0 is out of range (0-100)
  - Item 7: 'None' is not a number

Average of valid scores: 82.0

Nonfinite example:
Processed 4 items:
  Valid: 2
  Errors: 2

Error details:
  - Item 1: nan is not finite
  - Item 3: inf is not finite

Average of valid scores: 88.5


[85.0, 92.0]

---
### ⏱️ Checkpoint 4 of 5 — finally (target 03:55)

Does `finally` normally run whether an exception occurs or not? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [29]:
checkpoint_4_answer = ""  # enter your answer
check_answer(
    4, checkpoint_4_answer, 'yes',
    '`finally` is for cleanup that should always occur.',
)


Checkpoint 4: enter your prediction, then run again.


False

---
## Exercises — Problems to Solve

> Each exercise is a problem. Think about the **STEPS** before you code. Decompose the problem, plan your approach, then implement it.

Complete the exercises below. Each exercise cell starts with `# ✏️ [EXn]` — **do not remove this line**.

### Core Practice and Optional Extension

- **Exercises 1–8:** core in-class practice.
- **Exercises 9 and above:** optional extension; these are not homework.
- At Checkpoint 5, list the exercises you have tested and can explain. This is a self-report, not automatic grading.


---
## Exercises

Complete the exercises below. Each exercise cell starts with `# ✏️ [EXn]` — **do not remove this line**.

### Exercise 1: Catch Division Error (Easy)

Write a function `safe_divide(a, b)` that:
- Returns the result of `a / b`
- Catches `ZeroDivisionError` and returns the string `"Error: division by zero"`

**Expected output:**
```
safe_divide(10, 3) = 3.3333333333333335
safe_divide(10, 0) = Error: division by zero
safe_divide(0, 5) = 0.0
```

<details><summary>💡 Hint</summary>

Use `try: return a / b` and `except ZeroDivisionError: return "Error: division by zero"`. Remember, dividing 0 by 5 is valid (result is 0.0).

</details>

In [30]:
# ✏️ [EX1]
# Write your safe_divide function below.


### Exercise 2: Safe Integer Input (Easy)

Write a function `safe_int(text)` that:
- Tries to convert `text` to an integer
- Returns the integer if successful
- Returns `None` if it fails (catching `ValueError`)
- Prints a message when conversion fails

**Expected output:**
```
safe_int('42') = 42
safe_int('3.14') = None  (printed: Cannot convert '3.14' to integer)
safe_int('hello') = None  (printed: Cannot convert 'hello' to integer)
safe_int('0') = 0
```

<details><summary>💡 Hint</summary>

Use `try: return int(text)` and `except ValueError: print(...); return None`. Note that `int("3.14")` also raises `ValueError` because `int()` doesn't handle decimal strings directly.

</details>

In [31]:
# ✏️ [EX2]
# Write your safe_int function below.


### Exercise 3: Safe List Access (Medium)

Write a function `safe_get(lst, index, default=None)` that:
- Returns `lst[index]` if the index is valid
- Returns `default` if an `IndexError` occurs
- Returns `default` if a `TypeError` occurs (e.g., index is a string)

**Expected output:**
```
safe_get([10, 20, 30], 1) = 20
safe_get([10, 20, 30], 10) = None
safe_get([10, 20, 30], -1) = 30
safe_get([10, 20, 30], 'a') = None
safe_get([10, 20, 30], 10, default=-1) = -1
```

<details><summary>💡 Hint</summary>

Use `try: return lst[index]` and `except (IndexError, TypeError): return default`. Remember that negative indices are valid in Python (`-1` means last element).

</details>

In [32]:
# ✏️ [EX3]
# Write your safe_get function below.


### Exercise 4: Calculator with Error Handling (Medium)

Write a function `calc(expression)` that takes a string like `"10 + 3"` and:
- Splits it into number, operator, number
- Supports `+`, `-`, `*`, `/`
- Handles `ValueError` (non-numeric values)
- Handles `ZeroDivisionError` (division by zero)
- Returns a helpful error message for each case

**Expected output:**
```
calc('10 + 3') = 13.0
calc('10 / 0') = Error: division by zero
calc('abc + 3') = Error: invalid number
calc('10 % 3') = Error: unknown operator '%'
```

<details><summary>💡 Hint</summary>

Use `expression.split()` to get three parts. Convert the first and third parts to `float()` inside a `try` block. Check the operator with `if/elif`. Handle each error type separately.

</details>

**Finite-number rule:** after numeric conversion, reject `nan`, `inf`, and `-inf` with `math.isfinite()`. For an average with no valid numbers, return the documented empty result. Boolean values are flags, not measurements, in these validation exercises.


In [33]:
# ✏️ [EX4]
# Write your calc function below.


### Exercise 5: Safe Division Function (Medium)

Write a function `divide_many(numbers)` that takes a list of numbers and divides them sequentially (first / second / third / ...).

- Handle empty list (return `None`)
- Handle single item list (return that item)
- Handle `ZeroDivisionError` — skip zeros and print a warning
- Handle `TypeError` — skip non-numbers and print a warning

**Expected output:**
```
divide_many([100, 2, 5]) = 10.0
divide_many([100, 0, 5]) = 20.0  (printed: Skipping division by zero)
divide_many([100, 'a', 5]) = 20.0  (printed: Skipping non-number: a)
divide_many([]) = None
```

<details><summary>💡 Hint</summary>

Start with `result = numbers[0]`. Loop through `numbers[1:]`. For each number, try `result = result / number`. Catch `ZeroDivisionError` and `TypeError` separately, print warnings, and continue.

</details>

**Initial value:** for this exercise the first item must be a finite number. If it is invalid, return `None` with a message; only later invalid divisors are skipped. An initial zero is valid.


In [34]:
# ✏️ [EX5]
# Write your divide_many function below.


### Exercise 6: Validate and Convert (Medium)

Write a function `convert_all(values, target_type)` that converts a list of values to the specified type (`"int"`, `"float"`, or `"str"`).

- Return a list of successfully converted values
- Print a message for each failed conversion
- Count and report successes and failures

**Expected output:**
```
convert_all(['1', '2.5', 'abc', '4'], 'int')
  '1' → 1 ✓
  '2.5' → Failed (invalid literal for int()...)
  'abc' → Failed (invalid literal for int()...)
  '4' → 4 ✓
  Results: 2 succeeded, 2 failed
  Converted: [1, 4]
```

<details><summary>💡 Hint</summary>

Use a dictionary to map type names to functions: `{"int": int, "float": float, "str": str}`. Loop through values, try converting each one, and keep track of successes and failures.

</details>

In [35]:
# ✏️ [EX6]
# Write your convert_all function below.


### Exercise 7: Robust Average (Medium)

Write a function `robust_average(data)` that:
- Handles an empty list (return `0` with a message)
- Handles non-list input (return `0` with a message)
- Skips non-numeric values with a warning
- Returns the average of valid numbers

**Expected output:**
```
robust_average([10, 20, 'abc', 30, None, 40]) = 25.0
  (printed: Skipping non-numeric: 'abc', Skipping non-numeric: None)
  (4 valid values out of 6)
robust_average([]) = 0
  (printed: Empty list, returning 0)
robust_average('hello') = 0
  (printed: Expected a list, got str)
```

<details><summary>💡 Hint</summary>

Check `isinstance(data, list)` first. Filter valid numbers by trying `float(item)` for each item. Keep a list of valid numbers and count skipped items. If no valid numbers remain, return 0.

</details>

**Finite-number rule:** after numeric conversion, reject `nan`, `inf`, and `-inf` with `math.isfinite()`. For an average with no valid numbers, return the documented empty result. Boolean values are flags, not measurements, in these validation exercises.


In [36]:
# ✏️ [EX7]
# Write your robust_average function below.


### Exercise 8: Temperature Converter with Full Validation (Medium)

Write a function `convert_temperature(value, from_unit, to_unit)` that:
- Converts between "C" (Celsius), "F" (Fahrenheit), and "K" (Kelvin)
- Validates that `value` is a number
- Validates that units are "C", "F", or "K"
- Checks that temperature is above absolute zero (-273.15°C, -459.67°F, 0K)
- Returns the converted value or an error message

**Expected output:**
```
convert_temperature(100, 'C', 'F') = 212.0
convert_temperature(32, 'F', 'C') = 0.0
convert_temperature(-300, 'C', 'K') = Error: below absolute zero
convert_temperature(100, 'C', 'X') = Error: unknown unit 'X'
convert_temperature('hot', 'C', 'F') = Error: value must be a number
```

<details><summary>💡 Hint</summary>

First validate the value with `try: value = float(value)`. Then check if units are valid. Convert to Celsius first (as a common base), check absolute zero, then convert from Celsius to the target unit.

</details>

**Finite-number rule:** after numeric conversion, reject `nan`, `inf`, and `-inf` with `math.isfinite()`. For an average with no valid numbers, return the documented empty result. Boolean values are flags, not measurements, in these validation exercises.

**Boundary:** absolute zero itself is permitted (Kelvin ≥0). Use Celsius as the common intermediate unit and test the boundary as well as ordinary temperatures.


In [37]:
# ✏️ [EX8]
# Write your convert_temperature function below.


---
### Checkpoint 5 of 5 — Practice reflection (target 04:45)

After Exercises 1–8, edit `practiced_exercises` in the next cell.
List only the exercise numbers whose results you have tested and whose steps you can explain.
Leave the list empty until you have done that work. This is your explicit self-report;
the tool does not inspect or grade your solution and does not count execution history.

**Türkçe:** Bu liste öz değerlendirmedir. Hücreyi çalıştırmak tek başına yeterli değildir;
sonucu kontrol et ve çözüm adımlarını açıklayabildiğinden emin ol.


In [ ]:
# Add an exercise number only after testing and explaining your own work.
# Example: [1, 2] records your reflection about Exercises 1 and 2.
# Türkçe: Bu liste öz değerlendirmedir; kodunuzun doğruluğunu otomatik ölçmez.
practiced_exercises = []
exercise_checkpoint(5, practiced_exercises, expected=8)
show_progress_summary()


---
## 🌟 Optional Extension

Exercises 9 and above are optional enrichment. Stop here if the five-hour class has ended.


### Exercise 9: Grade Entry System (Medium)

Write a function `enter_grades(grade_strings)` that takes a list of grade strings and:
- Converts each to an integer
- Validates that each grade is between 0 and 100
- Returns a dictionary with:
  - `"valid"`: list of valid grades
  - `"invalid"`: list of `(original_value, reason)` tuples
  - `"average"`: average of valid grades (or 0 if none)

**Expected output:**
```
Results:
  Valid grades: [85, 92, 78, 100, 0]
  Invalid entries:
    'abc' - not a number
    '-5' - out of range (0-100)
    '150' - out of range (0-100)
  Average: 71.0
```

<details><summary>💡 Hint</summary>

Loop through each string. Use `try/except ValueError` for the conversion. Use `if grade < 0 or grade > 100` for range checking. Append to either the valid or invalid list accordingly.

</details>

In [39]:
# ✏️ [EX9]
# Write your enter_grades function below.
# Test with: ["85", "abc", "92", "-5", "78", "150", "100", "0"]


### Exercise 10: Multiple Error Types (Medium)

Write a function `process_record(record)` that takes a dictionary and extracts student information. Handle each error type with a specific message.

The function should:
- Access `record["name"]` — handle `KeyError`
- Access `record["grades"][0]` — handle `IndexError`
- Calculate `sum(record["grades"]) / len(record["grades"])` — handle `ZeroDivisionError`
- Convert `record["id"]` to `int` — handle `ValueError`

**Test records and expected output:**
```python
records = [
    {"name": "Elif", "id": "12345", "grades": [85, 92, 78]},
    {"id": "12346", "grades": [90]},           # missing name
    {"name": "Cem", "id": "abc", "grades": [75]}, # bad id
    {"name": "Deniz", "id": "12348", "grades": []}, # empty grades
]
```
```
Elif (12345): avg = 85.0 ✓
KeyError: 'name' is missing
Cem: ValueError: invalid ID 'abc'
Deniz (12348): ZeroDivisionError: no grades to average
```

<details><summary>💡 Hint</summary>

Use multiple `try/except` blocks, one for each operation. Or use nested tries. Access the name first, then the ID, then calculate the average. Each step has its own potential error.

</details>

**Operation order:** check name, ID, first-grade access, then average. Catch an empty-list `IndexError` locally and continue to the average check; the final returned message for an empty list is the documented `ZeroDivisionError: no grades to average`. This lets both operations be demonstrated without one hiding the other.


In [40]:
# ✏️ [EX10]
# Write your process_record function below.


### Exercise 11: Retry Pattern (Challenge)

Write a function `get_valid_input(prompt, validator, max_tries=3)` that:
- Takes a prompt string, a validator function, and max attempts
- The validator function takes a string and returns `(True, converted_value)` or `(False, error_message)`
- Keeps asking until valid input or max tries reached
- Returns the valid value or `None`

Create these validators:
- `validate_positive_int(text)` — must be a positive integer
- `validate_grade(text)` — must be integer 0-100
- `validate_name(text)` — must be at least 2 words, letters only

Since we can't use real `input()`, simulate it with a list of test inputs.

**Expected output:**
```
Testing positive int validator:
  'abc' → Invalid: not a valid integer
  '-5' → Invalid: must be positive
  '42' → Valid: 42

Testing grade validator:
  '150' → Invalid: must be between 0 and 100
  'abc' → Invalid: not a valid integer
  '85' → Valid: 85
```

<details><summary>💡 Hint</summary>

Each validator uses `try/except` internally. For example, `validate_positive_int` tries `int(text)`, catches `ValueError`, then checks `> 0`. The main function loops up to `max_tries` times, calling the validator each time.

</details>

**Deterministic input:** add an `inputs` list parameter (or a small input-provider parameter) so tests do not wait for a keyboard. Stop when the list is exhausted or `max_tries` is reached. For names, each whitespace-separated word must contain letters only.


In [41]:
# ✏️ [EX11]
# Write your retry pattern with validators below.


### Exercise 12: Robust Data Processor (Challenge)

Write a function `process_student_records(records)` that takes a list of dictionaries containing student data with potentially messy/missing fields.

Each record should have: `name`, `id`, `midterm`, `final`

The function should:
- Skip records with missing required fields (print which fields are missing)
- Convert scores to numbers (handle invalid values)
- Validate scores are 0-100
- Calculate weighted average (midterm 40%, final 60%)
- Return a list of processed records and a summary

**Test data:**
```python
records = [
    {"name": "Ayse Kara", "id": "2024001", "midterm": "85", "final": "90"},
    {"name": "Berk Yilmaz", "id": "2024002", "midterm": "abc", "final": "75"},
    {"id": "2024003", "midterm": "70", "final": "80"},
    {"name": "Ceren Demir", "id": "2024004", "midterm": "92", "final": "150"},
    {"name": "Doruk Ozkan", "id": "2024005", "midterm": "78", "final": "82"},
]
```

**Expected output (approximate):**
```
Processing 5 records...
✓ Ayse Kara: 88.0 (BA)
✗ Berk Yilmaz: invalid midterm score 'abc'
✗ Record 3: missing field 'name'
✗ Ceren Demir: final score 150 out of range
✓ Doruk Ozkan: 80.4 (BA)

Summary: 2 processed, 3 errors
```

<details><summary>💡 Hint</summary>

For each record: (1) check if all required keys exist using `"key" in record`, (2) try converting midterm and final to `float`, (3) validate range 0-100, (4) calculate weighted average. Use `try/except` for conversions and collect error messages.

</details>

**Finite-number rule:** after numeric conversion, reject `nan`, `inf`, and `-inf` with `math.isfinite()`. For an average with no valid numbers, return the documented empty result. Boolean values are flags, not measurements, in these validation exercises.


> **Example data only:** These fictional calculation weights are for this programming exercise. The course grade uses only the midterm (50%) and final (50%).


In [42]:
# ✏️ [EX12]
# Write your robust data processor below.


### Bridge Exercise: Preview of File I/O

Predict and catch two file-related exceptions without stopping the notebook:

1. In a new temporary working directory, try to read `missing_example.txt` and catch
   `FileNotFoundError`. A file must exist before it can be read.
2. Demonstrate a denied write **by simulation**: `raise PermissionError("simulated denied write")`
   inside `try/except PermissionError`. Do not try to write to system files.

**Expected output:** `FileNotFoundError: example file is absent`, then
`PermissionError: simulated denied write`.

The second exception is deliberately raised for a repeatable lesson: operating-system
permissions differ between computers. Next week you will read and write your own files.

**Türkçe:** Dosya yokluğu ile erişim izni hatasını ayrı yakalayın. İzin hatasını
göstermek için sistem dosyasını değiştirmek gerekmez; burada hatayı kontrollü üretiriz.


In [43]:
# ✏️ [EXBridge]
# Experiment with file-related errors.
# Use try/except to catch FileNotFoundError and PermissionError.


## Worked solutions and study support

Try each problem first. Then compare your reasoning and test cases with the complete
[Week 11 worked solutions](../solutions/Week_11_Solutions.ipynb).
The solution notebook includes every core exercise, optional exercise, and this week's bridge when present.

If you are using Colab, [open the published solution notebook](https://colab.research.google.com/github/ArifSolmaz/courses/blob/main/fall/cp1/solutions/Week_11_Solutions.ipynb).
Open it in a separate runtime. Running a solution first should not supply hidden variables to your own work.

**Türkçe:** Önce kendi çözümünü dene. Sonra adımları ve testleri karşılaştır; çözümü kapatıp farklı bir örneği kendin çöz.

[Simple course guide](../STUDY_GUIDE.md) · [All worked solutions](../solutions/README.md)
